# Explainable Rice Supply Forecasting Using LSTM and Internal RAG

**Studi Kasus: Pasokan Beras Karawang (2011–2025)**

Notebook ini mengimplementasikan pipeline lengkap:
1. **LSTM Forecasting** — Memprediksi pasokan beras Karawang 12 bulan ke depan
2. **Internal RAG** — Knowledge base dibangun dari dataset itu sendiri (rata-rata bulanan, tren tahunan, insight musiman)
3. **Gemini sebagai LLM** — Menghasilkan penjelasan (Explainable Forecast) berdasarkan knowledge yang diretriever dari Vector Database
4. **RAGAS Evaluation** — Mengukur kualitas RAG (faithfulness, relevance)

Framework:
```
Rice Supply Dataset → Preprocessing → LSTM Forecasting → Knowledge Extraction
    → Embedding → ChromaDB → Retriever → Gemini → Explainable Forecast → RAGAS
```

---
**Posisi Penelitian:**
*Explainable Rice Supply Forecasting Using LSTM and Internal Retrieval-Augmented Generation (RAG): A Case Study of Karawang Rice Supply (2011–2025)*

## Setup Environment

Jalankan sel berikut untuk menginstal dependensi yang diperlukan.

In [ ]:
# 1. Install dependencies
!pip install -q \
    pandas numpy matplotlib seaborn \
    tensorflow \
    sentence-transformers \
    chromadb \
    openai \
    python-dotenv \
    scikit-learn \
    datasets \
    google-cloud-aiplatform \
    "ragas>=0.1.0,<0.2.0"

print("\u2713 Dependencies installed")

## Konfigurasi Gemini API

Notebook ini menggunakan **Gemini API** (Google AI Studio) sebagai LLM untuk menghasilkan explainable forecast.

Dapatkan API key gratis di: https://aistudio.google.com/apikey

Jalankan sel berikut **satu per satu** (jangan Run All) — tunggu input box muncul sebelum menempelkan key.

In [ ]:
# 2. Setup Gemini API Key
import getpass
import os

while True:
    google_api_key = getpass.getpass("Enter your Google AI Studio API key: ").strip()
    if not google_api_key:
        print("\u274c Empty input \u2014 try again.")
        continue
    break

os.environ["GOOGLE_API_KEY"] = google_api_key
print(f"\u2713 API key saved (length: {len(google_api_key)})")

In [ ]:
# 2b. Verify Gemini API key works
from openai import OpenAI
from dotenv import load_dotenv

client = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

try:
    resp = client.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[{"role": "user", "content": "Say OK"}],
    )
    print("\u2713 Gemini API works:", resp.choices[0].message.content)
except Exception as e:
    print("\u274c Gemini API test failed:", e)
    print("Re-run the previous cell and paste the key again.")

---
## 1. Load Dataset

Dataset `ricesupply_2011-2025.csv` berisi data pasokan beras bulanan dari berbagai wilayah.
Target penelitian: **Karawang**

In [ ]:
# 3. Load dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams.update({'figure.max_open_warning': 0})

# Mount Google Drive to access dataset
from google.colab import drive
drive.mount('/content/drive')

# --- Sesuaikan path ini dengan lokasi file Anda ---
DATA_PATH = "/content/drive/MyDrive/ricesupply_2011-2025.csv"

df = pd.read_csv(DATA_PATH, index_col=0)
df['Bulan'] = pd.to_datetime(df['Bulan'])
df.set_index('Bulan', inplace=True)

print(f"Dataset shape: {df.shape}")
print(f"Periode: {df.index.min()} s.d. {df.index.max()}")
print(f"Kolom: {list(df.columns)}")
df.head()

---
## 2. Exploratory Data Analysis (EDA)

In [ ]:
# 4. EDA - Statistik deskriptif
target_col = "Karawang"
print("Statistik Deskriptif - Karawang:")
print("=" * 50)
stats = df[target_col].describe()
print(f"Count     : {stats['count']:.0f}")
print(f"Mean      : {stats['mean']:.2f} ton")
print(f"Std       : {stats['std']:.2f} ton")
print(f"Min       : {stats['min']:.2f} ton")
print(f"25%       : {stats['25%']:.2f} ton")
print(f"50%       : {stats['50%']:.2f} ton")
print(f"75%       : {stats['75%']:.2f} ton")
print(f"Max       : {stats['max']:.2f} ton")

print(f"\nTotal pasokan: {df[target_col].sum():,.0f} ton")

In [ ]:
# 5. Plot time series Karawang
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df.index, df[target_col], color='#2E86AB', linewidth=1.5, marker='o', markersize=3)
ax.set_title(f'Pasokan Beras Karawang ({df.index.year.min()} \u2013 {df.index.year.max()})', fontsize=14, fontweight='bold')
ax.set_xlabel('Tahun')
ax.set_ylabel('Pasokan (ton)')
ax.axhline(y=df[target_col].mean(), color='red', linestyle='--', alpha=0.7, label=f'Rata-rata: {df[target_col].mean():.0f} ton')
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. Analisis Musiman (Seasonality Analysis)

Menganalisis pola musiman pasokan beras Karawang.

In [ ]:
# 6. Monthly seasonality pattern
df_monthly = df.copy()
df_monthly['Month'] = df_monthly.index.month
df_monthly['Year'] = df_monthly.index.year

# Average by month across years
monthly_avg = df_monthly.groupby('Month')[target_col].agg(['mean', 'std', 'min', 'max'])
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'Mei', 'Jun', 'Jul', 'Agu', 'Sep', 'Okt', 'Nov', 'Des']
monthly_avg.index = month_names

print("Rata-rata Pasokan Bulanan (Karawang):")
print("=" * 60)
print(monthly_avg.to_string())

peak_month = monthly_avg['mean'].idxmax()
low_month = monthly_avg['mean'].idxmin()
print(f"\nBulan puncak: {peak_month} ({monthly_avg.loc[peak_month, 'mean']:.0f} ton)")
print(f"Bulan terendah: {low_month} ({monthly_avg.loc[low_month, 'mean']:.0f} ton)")

In [ ]:
# 7. Plot seasonality
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Monthly averages with error bars
ax = axes[0]
months_num = range(1, 13)
ax.errorbar(months_num, monthly_avg['mean'].values, yerr=monthly_avg['std'].values,
            capsize=5, capthick=1.5, marker='s', markersize=6,
            color='#2E86AB', linewidth=2, ecolor='gray', elinewidth=1)
ax.set_title('Rata-rata Pasokan per Bulan (dengan Std Dev)', fontsize=13, fontweight='bold')
ax.set_xlabel('Bulan')
ax.set_ylabel('Pasokan (ton)')
ax.set_xticks(months_num)
ax.set_xticklabels(month_names, rotation=45)
ax.grid(True, alpha=0.3)

# Plot 2: Boxplot by month
ax2 = axes[1]
sns.boxplot(data=df_monthly, x='Month', y=target_col, ax=ax2, palette='viridis')
ax2.set_title('Distribusi Pasokan per Bulan', fontsize=13, fontweight='bold')
ax2.set_xlabel('Bulan')
ax2.set_ylabel('Pasokan (ton)')
ax2.set_xticklabels(month_names, rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# 8. Yearly trend analysis
yearly = df_monthly.groupby('Year')[target_col].agg(['sum', 'mean', 'max', 'min'])

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(yearly.index, yearly['sum'], color='#2E86AB', alpha=0.8, edgecolor='white')
ax.set_title('Total Pasokan Tahunan - Karawang', fontsize=13, fontweight='bold')
ax.set_xlabel('Tahun')
ax.set_ylabel('Total Pasokan (ton)')

# Annotate bars
for i, v in enumerate(yearly['sum']):
    ax.text(yearly.index[i], v + 500, f'{v:,.0f}', ha='center', fontsize=8, rotation=45)

plt.tight_layout()
plt.show()

print("Statistik Tahunan:")
print(yearly.to_string())

---
## 4. LSTM Forecasting

Membangun model LSTM untuk memprediksi pasokan beras Karawang.

In [ ]:
# 9. Prepare data for LSTM
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

series = df[target_col].values.astype(float).reshape(-1, 1)

# Normalize
scaler = MinMaxScaler(feature_range=(0, 1))
series_scaled = scaler.fit_transform(series)

# Train-test split (80-20)
train_size = int(len(series_scaled) * 0.8)
train_data = series_scaled[:train_size]
test_data = series_scaled[train_size:]

print(f"Total data: {len(series_scaled)}")
print(f"Train: {len(train_data)}")
print(f"Test: {len(test_data)}")

# Create sequences for LSTM
def create_sequences(data, seq_length=12):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length, 0])
        y.append(data[i+seq_length, 0])
    return np.array(X), np.array(y)

SEQ_LENGTH = 12  # Use 12 months to predict next

X_train, y_train = create_sequences(train_data, SEQ_LENGTH)
X_test, y_test = create_sequences(test_data, SEQ_LENGTH)

# Reshape for LSTM: (samples, timesteps, features)
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

In [ ]:
# 10. Build and train LSTM model
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(SEQ_LENGTH, 1)),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

# Early stopping
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

# Train
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=8,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# 11. Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history.history['loss'], label='Train Loss', color='#2E86AB')
axes[0].plot(history.history['val_loss'], label='Val Loss', color='#A23B72')
axes[0].set_title('Model Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['mae'], label='Train MAE', color='#2E86AB')
axes[1].plot(history.history['val_mae'], label='Val MAE', color='#A23B72')
axes[1].set_title('Model MAE')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5. Evaluasi (RMSE, MAE, MAPE)

In [ ]:
# 12. Evaluate model
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Predict on test set
y_pred_scaled = model.predict(X_test)

# Inverse transform
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
y_pred_actual = scaler.inverse_transform(y_pred_scaled).flatten()

# Metrics
rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
mae = mean_absolute_error(y_test_actual, y_pred_actual)
mape = np.mean(np.abs((y_test_actual - y_pred_actual) / (y_test_actual + 1e-8))) * 100

print("Hasil Evaluasi:")
print("=" * 40)
print(f"RMSE: {rmse:.2f} ton")
print(f"MAE : {mae:.2f} ton")
print(f"MAPE: {mape:.2f}%")

In [ ]:
# 13. Plot actual vs predicted
fig, ax = plt.subplots(figsize=(16, 5))

# Test indices
test_idx = df.index[train_size + SEQ_LENGTH:]

ax.plot(test_idx, y_test_actual, label='Actual', color='#2E86AB', linewidth=1.5, marker='o', markersize=4)
ax.plot(test_idx, y_pred_actual, label='Predicted', color='#A23B72', linewidth=1.5, marker='s', markersize=4, linestyle='--')
ax.set_title('Actual vs Predicted - Pasokan Beras Karawang', fontsize=14, fontweight='bold')
ax.set_xlabel('Tanggal')
ax.set_ylabel('Pasokan (ton)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Residual plot
residuals = y_test_actual - y_pred_actual
fig, ax = plt.subplots(figsize=(16, 3))
ax.bar(test_idx, residuals, color=['#A23B72' if r < 0 else '#2E86AB' for r in residuals], alpha=0.7)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.set_title('Residuals (Actual - Predicted)', fontsize=12)
ax.set_ylabel('Residual (ton)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. Forecast 12 Bulan ke Depan

In [ ]:
# 14. Forecast next 12 months
def forecast_future(model, last_sequence, scaler, n_steps=12):
    """Generate n-step ahead forecast using iterative prediction."""
    current_seq = last_sequence.copy()
    predictions = []
    
    for _ in range(n_steps):
        # Reshape for prediction
        x = current_seq.reshape(1, SEQ_LENGTH, 1)
        pred_scaled = model.predict(x, verbose=0)[0, 0]
        predictions.append(pred_scaled)
        
        # Shift window: remove first element, append prediction
        current_seq = np.roll(current_seq, -1)
        current_seq[-1] = pred_scaled
    
    # Inverse transform
    predictions = np.array(predictions).reshape(-1, 1)
    return scaler.inverse_transform(predictions).flatten()

# Last sequence from full data
last_seq = series_scaled[-SEQ_LENGTH:].flatten()

# Forecast 12 months
forecast_values = forecast_future(model, last_seq, scaler, n_steps=12)

# Create date range for forecast
last_date = df.index[-1]
forecast_dates = pd.date_range(start=last_date + pd.DateOffset(months=1), periods=12, freq='MS')

forecast_df = pd.DataFrame({
    'Tanggal': forecast_dates,
    'Forecast (ton)': forecast_values.round(0).astype(int)
})
forecast_df.index = forecast_dates

print("Forecast 12 Bulan ke Depan:")
print("=" * 40)
for i, row in forecast_df.iterrows():
    print(f"{row['Tanggal'].strftime('%Y-%m')}: {row['Forecast (ton)']:>6.0f} ton")

In [ ]:
# 15. Plot historical + forecast
fig, ax = plt.subplots(figsize=(16, 5))

# Historical
ax.plot(df.index, df[target_col], label='Historical', color='#2E86AB', linewidth=1.5)

# Forecast
ax.plot(forecast_dates, forecast_values, label='Forecast', color='#F18F01', linewidth=2, marker='o', markersize=5)

# Vertical line separating history from forecast
ax.axvline(x=last_date, color='gray', linestyle='--', alpha=0.5)
ax.text(last_date, ax.get_ylim()[1]*0.95, 'Forecast Start', rotation=90, fontsize=10, color='gray')

ax.set_title('Pasokan Beras Karawang - Historical & Forecast', fontsize=14, fontweight='bold')
ax.set_xlabel('Tahun')
ax.set_ylabel('Pasokan (ton)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 7. Knowledge Extraction (Internal RAG)

Knowledge base dibangun dari dataset itu sendiri, bukan dari dokumen eksternal.

Knowledge yang diekstrak meliputi:
- Rata-rata bulanan
- Tren tahunan
- Bulan dengan pasokan tertinggi
- Bulan dengan pasokan terendah
- Statistik deskriptif
- Insight musiman

In [ ]:
# 16. Extract knowledge from dataset

knowledge_documents = []

# --- 1. Monthly averages ---
monthly_avg_data = df_monthly.groupby('Month')[target_col].agg(['mean', 'std', 'min', 'max'])
for month_num in range(1, 13):
    row = monthly_avg_data.loc[month_num]
    doc = {
        "id": f"monthly_avg_{month_num:02d}",
        "text": (
            f"Rata-rata pasokan beras Karawang pada bulan {month_names[month_num-1]} "
            f"adalah {row['mean']:.0f} ton (std: {row['std']:.0f}, min: {row['min']:.0f}, max: {row['max']:.0f})."
        ),
        "metadata": {"category": "rata_rata_bulanan", "month": month_num, "month_name": month_names[month_num-1]}
    }
    knowledge_documents.append(doc)

# --- 2. Yearly trends ---
yearly_data = df_monthly.groupby('Year')[target_col].agg(['sum', 'mean', 'min', 'max'])
for year in yearly_data.index:
    row = yearly_data.loc[year]
    doc = {
        "id": f"yearly_{year}",
        "text": (
            f"Tahun {year}: total pasokan beras Karawang {row['sum']:.0f} ton, "
            f"rata-rata bulanan {row['mean']:.0f} ton, "
            f"terendah {row['min']:.0f} ton, tertinggi {row['max']:.0f} ton."
        ),
        "metadata": {"category": "tren_tahunan", "year": int(year)}
    }
    knowledge_documents.append(doc)

# --- 3. Peak and low months overall ---
peak_row = df_monthly.loc[df_monthly[target_col].idxmax()]
low_row = df_monthly.loc[df_monthly[target_col].idxmin()]
knowledge_documents.append({
    "id": "peak_overall",
    "text": (
        f"Pasokan tertinggi sepanjang sejarah: {peak_row[target_col]:.0f} ton "
        f"pada {peak_row.name.strftime('%B %Y')}."
    ),
    "metadata": {"category": "insight_musiman", "type": "peak"}
})
knowledge_documents.append({
    "id": "low_overall",
    "text": (
        f"Pasokan terendah sepanjang sejarah: {low_row[target_col]:.0f} ton "
        f"pada {low_row.name.strftime('%B %Y')}."
    ),
    "metadata": {"category": "insight_musiman", "type": "low"}
})

# --- 4. Seasonal insight ---
knowledge_documents.append({
    "id": "seasonal_pattern",
    "text": (
        f"Analisis musiman menunjukkan bahwa pasokan beras Karawang memiliki pola yang konsisten. "
        f"Titik terendah terjadi pada bulan Januari-Februari (rata-rata {monthly_avg_data.loc[1, 'mean']:.0f} ton "
        f"dan {monthly_avg_data.loc[2, 'mean']:.0f} ton), "
        f"sedangkan puncak terjadi pada bulan April-Mei (rata-rata {monthly_avg_data.loc[4, 'mean']:.0f} ton "
        f"dan {monthly_avg_data.loc[5, 'mean']:.0f} ton). "
        f"Pola ini sejalan dengan musim panen di Karawang."
    ),
    "metadata": {"category": "insight_musiman", "type": "seasonal_pattern"}
})

# --- 5. Descriptive statistics ---
desc = df[target_col].describe()
knowledge_documents.append({
    "id": "descriptive_stats",
    "text": (
        f"Statistik deskriptif pasokan beras Karawang ({df.index.year.min()}-{df.index.year.max()}): "
        f"rata-rata {desc['mean']:.0f} ton, standar deviasi {desc['std']:.0f} ton, "
        f"minimum {desc['min']:.0f} ton, maksimum {desc['max']:.0f} ton, "
        f"median {desc['50%']:.0f} ton. Total pasokan {df[target_col].sum():,.0f} ton."
    ),
    "metadata": {"category": "statistik_deskriptif"}
})

# --- 6. Forecast knowledge ---
for i, (date, val) in enumerate(zip(forecast_dates, forecast_values)):
    knowledge_documents.append({
        "id": f"forecast_{date.strftime('%Y%m')}",
        "text": (
            f"Prediksi pasokan beras Karawang untuk {date.strftime('%B %Y')} "
            f"adalah {val:.0f} ton."
        ),
        "metadata": {"category": "forecast", "date": date.strftime('%Y-%m-%d')}
    })

print(f"\u2713 Extracted {len(knowledge_documents)} knowledge documents")
print("\nSample documents:")
for doc in knowledge_documents[:5]:
    print(f"  [{doc['id']}] {doc['text'][:100]}...")

---
## 8. Embedding & ChromaDB

Knowledge yang telah diekstrak di-embedding menggunakan Sentence Transformer dan disimpan di ChromaDB.

In [ ]:
# 17. Initialize ChromaDB with Sentence Transformer embeddings
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

# Initialize embedding model
print("Loading Sentence Transformer model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("\u2713 Model loaded")

# Initialize ChromaDB with persistence
chroma_client = chromadb.Client(Settings(
    persist_directory="/content/rice_supply_chromadb",
    anonymized_telemetry=False
))

# Create or get collection
try:
    chroma_client.delete_collection("rice_supply_knowledge")
except:
    pass

collection = chroma_client.create_collection(
    name="rice_supply_knowledge",
    metadata={"hnsw:space": "cosine"}
)

# Embed and add documents
texts = [doc['text'] for doc in knowledge_documents]
ids = [doc['id'] for doc in knowledge_documents]
metadatas = [doc['metadata'] for doc in knowledge_documents]

# Generate embeddings
embeddings = embedding_model.encode(texts, show_progress_bar=True)

# Add to ChromaDB
collection.add(
    embeddings=embeddings.tolist(),
    documents=texts,
    metadatas=metadatas,
    ids=ids
)

print(f"\u2713 Added {len(texts)} documents to ChromaDB collection '{collection.name}'")
print(f"\u2713 ChromaDB persisted at /content/rice_supply_chromadb")

---
## 9. Retrieval

Fungsi retriever untuk mengambil knowledge yang relevan dari ChromaDB.

In [ ]:
# 18. Retrieval function
def retrieve_knowledge(query, k=5):
    """Retrieve relevant knowledge from ChromaDB."""
    query_embedding = embedding_model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=k
    )
    
    retrieved = []
    for i in range(len(results['documents'][0])):
        retrieved.append({
            'text': results['documents'][0][i],
            'metadata': results['metadatas'][0][i],
            'distance': results['distances'][0][i] if 'distances' in results else None
        })
    return retrieved

# Test retrieval
test_queries = [
    "Bagaimana pola musiman pasokan beras Karawang?",
    "Berapa prediksi pasokan untuk tahun depan?",
    "Kapan bulan dengan pasokan tertinggi?"
]

for q in test_queries:
    print(f"\nQuery: {q}")
    print("-" * 60)
    results = retrieve_knowledge(q, k=3)
    for r in results:
        print(f"  [{r['metadata'].get('category', 'unknown')}] {r['text'][:120]}...")

---
## 10. Explainable Forecast dengan Gemini

Menggunakan Gemini sebagai LLM untuk menghasilkan penjelasan (natural language explanation) berdasarkan:
1. Hasil prediksi LSTM
2. Knowledge yang diretriever dari Vector Database

In [ ]:
# 19. Generate explainable forecast using Gemini + RAG
def generate_explainable_forecast(forecast_values, forecast_dates, query=""):
    """
    Generate explainable forecast using Gemini with RAG context.
    
    Args:
        forecast_values: Array of forecast values
        forecast_dates: Array of forecast dates
        query: Optional specific question from user
    
    Returns:
        dict: Contains explanation and metadata
    """
    # Prepare forecast summary
    forecast_summary = []
    for date, val in zip(forecast_dates, forecast_values):
        forecast_summary.append(f"- {date.strftime('%B %Y')}: {val:.0f} ton")
    forecast_str = "\n".join(forecast_summary)
    
    avg_forecast = np.mean(forecast_values)
    min_forecast = np.min(forecast_values)
    max_forecast = np.max(forecast_values)
    
    # Retrieve relevant knowledge
    primary_query = query or f"Forecast pasokan beras Karawang {forecast_dates[0].year}"
    retrieved = retrieve_knowledge(primary_query, k=5)
    context_str = "\n\n".join([r['text'] for r in retrieved])
    
    # Build prompt for Gemini
    system_prompt = (
        "Anda adalah ahli pertanian dan supply chain yang mengkhususkan diri pada analisis "
        "pasokan beras di Indonesia. Tugas Anda adalah memberikan penjelasan yang komprehensif "
        "dan mudah dipahami tentang hasil prediksi pasokan beras.\n\n"
        "Gunakan knowledge yang disediakan untuk mendukung penjelasan Anda. "
        "Bandingkan hasil prediksi dengan pola historis jika relevan. "
        "Berikan interpretasi yang bermakna, bukan sekadar mengulang angka."
    )
    
    user_prompt = f"""
## Knowledge Base (dari dataset historis):
{context_str}

## Hasil Prediksi LSTM (12 bulan ke depan):
{forecast_str}

Ringkasan statistik forecast:
- Rata-rata: {avg_forecast:.0f} ton
- Terendah: {min_forecast:.0f} ton
- Tertinggi: {max_forecast:.0f} ton

## Tugas:
Berdasarkan knowledge di atas dan hasil prediksi LSTM, berikan penjelasan yang mencakup:
1. **Interpretasi Pola**: Bagaimana pola prediksi dibandingkan dengan pola musiman historis?
2. **Analisis Angka**: Apa arti angka-angka prediksi tersebut dalam konteks historis?
3. **Insight**: Insight apa yang bisa diambil dari perbandingan prediksi dengan data historis?
4. **Rekomendasi**: Rekomendasi berdasarkan hasil prediksi.
"""
    
    # Call Gemini via OpenAI-compatible endpoint
    response = client.chat.completions.create(
        model="gemini-2.5-flash",
        temperature=0.7,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    
    explanation = response.choices[0].message.content
    
    return {
        "forecast": forecast_str,
        "explanation": explanation,
        "retrieved_knowledge": retrieved,
        "usage": response.usage.model_dump() if response.usage else None
    }

In [ ]:
# 20. Run explainable forecast
print("Generating explainable forecast...")
result = generate_explainable_forecast(forecast_values, forecast_dates)

print("\n" + "=" * 70)
print("EXPLAINABLE FORECAST - Gemini + Internal RAG")
print("=" * 70)

print("\n--- Forecast ---")
print(result['forecast'])

print("\n--- Explanation ---")
print(result['explanation'])

print("\n--- Retrieved Knowledge Sources ---")
for r in result['retrieved_knowledge']:
    print(f"  [{r['metadata'].get('category', 'unknown')}] {r['text'][:100]}...")

---
## 11. RAGAS Evaluation

Mengukur kualitas RAG menggunakan metrik RAGAS (Retrieval Augmented Generation Assessment):
- **Faithfulness**: Apakah jawaban LLM setia pada context yang diberikan?
- **Answer Relevancy**: Seberapa relevan jawaban dengan pertanyaan?
- **Context Precision**: Seberapa presisi context yang diretriever?
- **Context Recall**: Seberapa lengkap context yang diretriever?

In [ ]:
# 21. RAGAS Evaluation

def _show_manual_eval(questions, answers, ground_truths):
    """Display manual evaluation comparison."""
    print("\n" + "=" * 70)
    print("MANUAL EVALUATION - Answers vs Ground Truth")
    print("=" * 70)
    for i, (q, a, gt) in enumerate(zip(questions, answers, ground_truths)):
        print(f"\nQ{i+1}: {q}")
        print(f"\n  Ground Truth: {gt}")
        print(f"\n  Answer: {a}")
        print("-" * 50)

# Prepare evaluation dataset
min_forecast_val = np.min(forecast_values)
max_forecast_val = np.max(forecast_values)

eval_questions = [
    "Bagaimana pola musiman pasokan beras Karawang?",
    "Berapa rata-rata pasokan beras Karawang per bulan?",
    "Kapan bulan puncak dan terendah pasokan beras Karawang?",
    "Bagaimana tren pasokan beras Karawang dari tahun ke tahun?",
    "Berapa prediksi pasokan beras Karawang untuk 12 bulan ke depan?"
]

eval_ground_truths = [
    "Pasokan beras Karawang memiliki pola musiman: terendah Januari-Februari dan puncak April-Mei, sejalan dengan musim panen.",
    f"Rata-rata pasokan bulanan bervariasi, terendah {monthly_avg_data.loc[1, 'mean']:.0f} ton (Januari) dan tertinggi {monthly_avg_data.loc[5, 'mean']:.0f} ton (Mei).",
    f"Bulan puncak adalah {peak_month} ({monthly_avg.loc[peak_month, 'mean']:.0f} ton) dan terendah {low_month} ({monthly_avg.loc[low_month, 'mean']:.0f} ton).",
    f"Total pasokan tahunan bervariasi, dengan rata-rata {df[target_col].mean():.0f} ton per bulan.",
    f"Prediksi 12 bulan ke depan berkisar antara {min_forecast_val:.0f} hingga {max_forecast_val:.0f} ton."
]

# Generate answers with RAG for each question
eval_answers = []
eval_contexts = []

for q in eval_questions:
    retrieved = retrieve_knowledge(q, k=3)
    contexts = [r['text'] for r in retrieved]
    eval_contexts.append(contexts)
    
    context_str = "\n\n".join(contexts)
    response = client.chat.completions.create(
        model="gemini-2.5-flash",
        temperature=0.3,
        messages=[
            {"role": "system", "content": "Jawab pertanyaan berdasarkan context yang diberikan secara akurat dan singkat."},
            {"role": "user", "content": f"Context:\n{context_str}\n\nPertanyaan: {q}"}
        ]
    )
    eval_answers.append(response.choices[0].message.content)
    print(f"\u2713 Answered: {q[:40]}...")

# Try RAGAS evaluation, fallback to manual display
try:
    from datasets import Dataset
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
    
    ragas_dataset = Dataset.from_dict({
        "question": eval_questions,
        "answer": eval_answers,
        "contexts": eval_contexts,
        "ground_truth": eval_ground_truths
    })
    
    print("\n" + "=" * 50)
    print("Evaluating with RAGAS...")
    print("=" * 50)
    
    ragas_result = evaluate(
        ragas_dataset,
        metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
        llm=client.chat.completions.create,
        embeddings=embedding_model.encode
    )
    
    print("\nRAGAS Evaluation Results:")
    print("=" * 50)
    print(ragas_result)
    
    ragas_df = ragas_result.to_pandas()
    print("\nPer-question scores:")
    display(ragas_df)
    
    print("\nSummary:")
    for col in ragas_df.select_dtypes(include=[np.number]).columns:
        print(f"  {col}: {ragas_df[col].mean():.4f}")

except ImportError as ie:
    print(f"\nNote: RAGAS import issue: {ie}")
    print("Showing manual evaluation results instead.")
    _show_manual_eval(eval_questions, eval_answers, eval_ground_truths)

except Exception as e:
    print(f"\nRAGAS evaluation failed: {e}")
    print("Showing manual evaluation results instead.")
    _show_manual_eval(eval_questions, eval_answers, eval_ground_truths)


---
## 12. Contoh Penggunaan Explainable Forecast

Contoh interaksi tanya-jawab dengan sistem Explainable Forecast.

In [ ]:
# 22. Interactive query example
user_queries = [
    "Mengapa prediksi bulan Januari lebih rendah dibanding bulan lainnya?",
    "Apakah tren pasokan Karawang meningkat atau menurun?",
    "Bandingkan prediksi tahun ini dengan tahun-tahun sebelumnya."
]

for q in user_queries:
    print("\n" + "=" * 70)
    print(f"Pertanyaan: {q}")
    print("=" * 70)
    
    result = generate_explainable_forecast(forecast_values, forecast_dates, query=q)
    print(f"\n{result['explanation']}")
    print(f"\n--- Knowledge digunakan: ---")
    for r in result['retrieved_knowledge']:
        print(f"  [{r['metadata'].get('category', 'unknown')}] {r['text'][:120]}")

---
## 13. Interactive Q&A (Manual Input)

Jalankan sel berikut untuk mengajukan pertanyaan Anda sendiri tentang forecast.

In [ ]:
# 23. Custom query
user_query = input("Masukkan pertanyaan Anda tentang forecast pasokan beras Karawang: ")

if user_query.strip():
    print("\n" + "=" * 70)
    result = generate_explainable_forecast(forecast_values, forecast_dates, query=user_query)
    print(result['explanation'])
else:
    print("Tidak ada pertanyaan yang dimasukkan.")

---
## Kesimpulan

Notebook ini mengimplementasikan:

1. **LSTM Forecasting** - Model deep learning untuk memprediksi pasokan beras Karawang
2. **Internal RAG** - Knowledge base yang dibangun dari dataset itu sendiri (bukan dokumen eksternal)
3. **ChromaDB + Sentence Transformer** - Vector database untuk penyimpanan dan pencarian knowledge
4. **Gemini LLM** - Menghasilkan explainable forecast berdasarkan knowledge yang diretriever
5. **RAGAS Evaluation** - Mengukur kualitas RAG (faithfulness, relevancy, precision, recall)

Kontribusi penelitian: memanfaatkan dataset time series sebagai sumber knowledge base untuk menjelaskan hasil prediksi, bukan hanya sebagai input model forecasting.

---
*RiceSupply_LSTM_InternalRAG.ipynb*